In [ ]:
from pyspark.sql.functions import col, lit, concat, avg, count, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType

StatementMeta(, 8b8ebd80-07a2-4f49-89f2-d59570f01eb9, 12, Finished, Available, Finished, False)

In [ ]:
# 1. LOAD SILVER DATA
df_silver = spark.table("silver_asset_health")

# 2. CREATE DIMENSION TABLE (Asset Master List)
df_dim_asset = df_silver.select(
    "AssetID", 
    "AssetType", 
    "Substation", 
    "InstallationDate"
).distinct() \
 .withColumn("MapLocation", concat(col("Substation"), lit(", New Zealand")))

StatementMeta(, 8b8ebd80-07a2-4f49-89f2-d59570f01eb9, 13, Finished, Available, Finished, False)

In [ ]:
# 3. CREATE FACT TABLE (Health Metrics)

df_fact_health = df_silver.select(
    "AssetID", 
    "ReadingTimestamp", 
    "Temperature_C", 
    "Load_Pct", 
    "OilLevel_Pct", 
    "Health_Score",
    "Health_Status"
)

StatementMeta(, 8b8ebd80-07a2-4f49-89f2-d59570f01eb9, 14, Finished, Available, Finished, False)

In [ ]:
# --- 4. INTEGRITY CHECK ---
orphan_count = df_fact_health.join(df_dim_asset, "AssetID", "left_anti").count()

if orphan_count == 0:
    print("✅ Integrity Check: Success. All asset readings match valid assets.")
    
   # --- 5. SAVE TO GOLD LAYER ---
    df_dim_asset.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_asset")
    df_fact_health.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_asset_readings")
    
    print("🚀 Gold Layer Tables 'dim_asset' and 'fact_asset_readings' are now LIVE.")
else:
    print(f"⚠️ Data Quality Alert: {orphan_count} readings do not have matching AssetIDs.") 

StatementMeta(, 8b8ebd80-07a2-4f49-89f2-d59570f01eb9, 15, Finished, Available, Finished, False)

✅ Integrity Check: Success. All asset readings match valid assets.
🚀 Gold Layer Tables 'dim_asset' and 'fact_asset_readings' are now LIVE.
